In [3]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.datasets import make_classification

### Generate Dataset

In [4]:
X, y = make_classification(n_samples=1000, n_features=10, n_informative=8, n_redundant=2, n_repeated=0, n_classes=2, random_state=42)

### Hyperparameter Tunning Using GridSearchCV

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=2000)

param_grid = {
    'C': [1, 2, 3, 4, 5],
    'solver': ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'],
}

grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, scoring='accuracy', verbose=1)

grid_search.fit(X_train, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best cross-validation accuracy: {:.2f}".format(grid_search.best_score_))

# Evaluate on test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("Test set accuracy: {:.2f}".format(accuracy_score(y_test, y_pred)))
print("Classification report:\n", classification_report(y_test, y_pred))
 

Fitting 3 folds for each of 25 candidates, totalling 75 fits
Best parameters found:  {'C': 1, 'solver': 'liblinear'}
Best cross-validation accuracy: 0.72
Test set accuracy: 0.68
Classification report:
               precision    recall  f1-score   support

           0       0.73      0.62      0.67       106
           1       0.63      0.73      0.68        94

    accuracy                           0.68       200
   macro avg       0.68      0.68      0.67       200
weighted avg       0.68      0.68      0.67       200



In [6]:
import optuna
from sklearn.model_selection import cross_val_score

c:\Office_Data\Prsonal data\Prsonal data\Resume\myrepo\GenAIAndMachineLearningProjects\PracticeCode\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
def objective(trial):
    C = trial.suggest_float('C', 1, 5)
    solver = trial.suggest_categorical('solver', ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'])
    
    model = LogisticRegression(C=C, solver=solver, max_iter=1000)
    
    accuracy = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    
    return accuracy

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

print("Best parameters found: ", study.best_params)
print("Best cross-validation accuracy: {:.2f}".format(study.best_value))

[I 2026-07-02 08:08:34,230] A new study created in memory with name: no-name-cbc88004-897f-4890-82c6-2f96c1a02966
[I 2026-07-02 08:08:34,255] Trial 0 finished with value: 0.7187115729398403 and parameters: {'C': 3.2791436451965303, 'solver': 'lbfgs'}. Best is trial 0 with value: 0.7187115729398403.
[I 2026-07-02 08:08:34,267] Trial 1 finished with value: 0.7187115729398403 and parameters: {'C': 3.574711051869826, 'solver': 'liblinear'}. Best is trial 0 with value: 0.7187115729398403.
[I 2026-07-02 08:08:34,285] Trial 2 finished with value: 0.71745844010776 and parameters: {'C': 1.1393900558431862, 'solver': 'newton-cg'}. Best is trial 0 with value: 0.7187115729398403.
[I 2026-07-02 08:08:34,295] Trial 3 finished with value: 0.7187115729398403 and parameters: {'C': 3.6570072516790426, 'solver': 'liblinear'}. Best is trial 0 with value: 0.7187115729398403.
[I 2026-07-02 08:08:34,312] Trial 4 finished with value: 0.7187115729398403 and parameters: {'C': 3.9930104255607413, 'solver': 'sag'

Best parameters found:  {'C': 1.487471503198699, 'solver': 'liblinear'}
Best cross-validation accuracy: 0.72


In [8]:
best_model_optuna = LogisticRegression(**study.best_params, max_iter=2000)
best_model_optuna.fit(X_train, y_train)
y_pred_optuna = best_model_optuna.predict(X_test)

report = classification_report(y_test, y_pred_optuna)
print("Test set accuracy: {:.2f}".format(accuracy_score(y_test, y_pred_optuna)))
print("Classification report:\n", report)

Test set accuracy: 0.68
Classification report:
               precision    recall  f1-score   support

           0       0.73      0.62      0.67       106
           1       0.63      0.73      0.68        94

    accuracy                           0.68       200
   macro avg       0.68      0.68      0.67       200
weighted avg       0.68      0.68      0.67       200

